In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Project path'i ekle
sys.path.append('../src')

# Custom modules
from model_evaluation import CreditScoreModelEvaluator, run_complete_evaluation

# Plot ayarları
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("📋 Model Evaluation Notebook")
print("="*50)
print("✅ Kütüphaneler yüklendi")
print("✅ Modüller import edildi")

📋 Model Evaluation Notebook
✅ Kütüphaneler yüklendi
✅ Modüller import edildi


In [4]:
from pathlib import Path
import pandas as pd

print("📂 Test verileri yükleniyor...")

# Proje yol yapısı
project_root = Path.cwd().parent  # credit-score-classifier/ dizinine çık
test_data_path = project_root / 'data' / 'processed' / 'test_processed.csv'

try:
    # Test verilerini yükle
    test_data = pd.read_csv(test_data_path)
    print("✅ Test verileri başarıyla yüklendi")
    
    # Hedef değişken ve özellikler ayır
    target_col = 'Credit_Score'
    
    # ID sütununu çıkar (varsa)
    feature_cols = [col for col in test_data.columns 
                   if col not in [target_col, 'ID', 'Customer_ID']]
    
    X_test = test_data[feature_cols]
    y_test = test_data[target_col]
    
    print(f"\n📊 Test Veri Bilgileri:")
    print(f"   Dosya yolu: {test_data_path}")
    print(f"   X_test shape: {X_test.shape}")
    print(f"   y_test shape: {y_test.shape}")
    print(f"   Özellik sayısı: {len(feature_cols)}")
    
    # Hedef sınıf dağılımı
    print(f"\n🎯 Hedef Sınıf Dağılımı:")
    class_counts = y_test.value_counts().sort_index()
    for class_val, count in class_counts.items():
        percentage = (count / len(y_test)) * 100
        print(f"   {class_val}: {count} ({percentage:.1f}%)")
    
    # İlk birkaç satırı göster
    print(f"\n🔍 X_test ilk 3 satır:")
    display(X_test.head(3))
    
    # Özellik isimleri
    feature_names = X_test.columns.tolist()
    print(f"\n📋 Özellik isimleri hazır: {len(feature_names)} adet")
    
except FileNotFoundError:
    print(f"❌ Test verisi bulunamadı: {test_data_path}")
    print("🔧 Önce preprocessing notebook'unu çalıştırın")
    
except Exception as e:
    print(f"❌ Veri yükleme hatası: {str(e)}")
    print("🔧 Veri dosyasını kontrol edin")

📂 Test verileri yükleniyor...
✅ Test verileri başarıyla yüklendi
❌ Veri yükleme hatası: 'Credit_Score'
🔧 Veri dosyasını kontrol edin


In [5]:
print("🚀 MODEL EVALUATION PIPELINE BAŞLATILIYOR")
print("="*60)

# Pipeline'ı çalıştır
evaluator, evaluation_results, comparison_df = run_complete_evaluation(
    X_test, y_test, 
    models_dir="../models/"
)

# Sonuçları kontrol et
if evaluator is not None and evaluation_results is not None:
    print("✅ Değerlendirme tamamlandı!")
    print(f"📊 {len(evaluation_results)} model değerlendirildi")
else:
    print("❌ Değerlendirme başarısız!")


🚀 MODEL EVALUATION PIPELINE BAŞLATILIYOR


NameError: name 'y_test' is not defined

In [ ]:
if comparison_df is not None:
    print("🏆 MODEL PERFORMANS KARŞILAŞTIRMASI")
    print("="*50)
    
    # Tabloyu göster
    display(comparison_df)
    
    # En iyi 3 model
    print("\n🥇 EN İYİ 3 MODEL:")
    top_3 = comparison_df.head(3)
    for idx, (_, row) in enumerate(top_3.iterrows(), 1):
        emoji = ["🥇", "🥈", "🥉"][idx-1]
        print(f"{emoji} {row['Model']}: {row['Test_f1_macro']:.4f}")
    
    # Model performans metrikleri detay
    print(f"\n📈 DETAYLI PERFORMANS METRİKLERİ:")
    for model_name, results in evaluation_results.items():
        print(f"\n{model_name}:")
        print(f"  • Accuracy: {results['accuracy']:.4f}")
        print(f"  • F1-Macro: {results['f1_macro']:.4f}")
        print(f"  • F1-Weighted: {results['f1_weighted']:.4f}")
        if results['roc_auc']:
            print(f"  • ROC-AUC: {results['roc_auc']:.4f}")
        if results['cv_score_mean']:
            print(f"  • CV Score: {results['cv_score_mean']:.4f} (±{results['cv_score_std']:.4f})")


In [ ]:

if evaluation_results:
    print("🔍 CONFUSION MATRIX ANALİZİ")
    print("="*40)
    
    # Her model için confusion matrix
    for model_name, results in evaluation_results.items():
        print(f"\n📊 {model_name} Confusion Matrix:")
        
        # Get test data (encoded)
        _, y_test_encoded = evaluator.prepare_test_data(X_test, y_test)
        
        # Plot confusion matrix
        cm, cm_normalized = evaluator.plot_confusion_matrix(
            y_test_encoded, results['y_pred'], model_name
        )
        
        # Sınıf bazında analiz
        print(f"\n📋 {model_name} Sınıf Bazında Performans:")
        class_names = ['Poor', 'Standard', 'Good']
        for i, class_name in enumerate(class_names):
            precision = results['precision'][i]
            recall = results['recall'][i]
            f1 = results['f1_scores'][i]
            support = results['support'][i]
            
            print(f"  {class_name}:")
            print(f"    Precision: {precision:.3f}")
            print(f"    Recall: {recall:.3f}")
            print(f"    F1-Score: {f1:.3f}")
            print(f"    Support: {support}")


In [ ]:
if evaluation_results:
    print("📈 ROC CURVE ANALİZİ")
    print("="*30)
    
    # Get test data (encoded)
    _, y_test_encoded = evaluator.prepare_test_data(X_test, y_test)
    
    # Her model için ROC curves
    for model_name, results in evaluation_results.items():
        if results['y_pred_proba'] is not None:
            print(f"\n🎯 {model_name} ROC Curves:")
            fpr, tpr, roc_auc = evaluator.plot_roc_curves(
                y_test_encoded, results['y_pred_proba'], model_name
            )
            
            # AUC skorları özet
            print(f"AUC Skorları:")
            class_names = ['Poor', 'Standard', 'Good']
            for i, class_name in enumerate(class_names):
                if i in roc_auc:
                    print(f"  {class_name}: {roc_auc[i]:.3f}")
            if 'micro' in roc_auc:
                print(f"  Micro-average: {roc_auc['micro']:.3f}")
        else:
            print(f"⚠️ {model_name}: Probability tahminleri mevcut değil")


In [ ]:

if evaluation_results:
    print("🔍 FEATURE IMPORTANCE ANALİZİ")
    print("="*40)
    
    feature_names = X_test.columns.tolist()
    
    # Tree-based modeller için feature importance
    tree_models = ['RandomForest', 'LightGBM', 'XGBoost']
    
    for model_name in tree_models:
        if model_name in evaluation_results:
            model = evaluation_results[model_name]['model']
            
            if hasattr(model, 'feature_importances_'):
                print(f"\n🌳 {model_name} Feature Importance:")
                
                # Feature importance plot
                feat_imp_df = evaluator.plot_feature_importance(
                    model, feature_names, model_name, top_n=15
                )
                
                # Top 10 özellikleri yazdır
                print(f"Top 10 En Önemli Özellikler:")
                top_10 = feat_imp_df.head(10)
                for idx, (_, row) in enumerate(top_10.iterrows(), 1):
                    print(f"  {idx:2d}. {row['feature']:<25}: {row['importance']:.4f}")
            else:
                print(f"⚠️ {model_name}: Feature importance mevcut değil")


In [ ]:
if evaluation_results and comparison_df is not None:
    print("📊 MODEL PERFORMANS GÖRSELLEŞTİRMESİ")
    print("="*45)
    
    # Farklı metrikler için karşılaştırma
    metrics_to_compare = ['f1_macro', 'f1_weighted', 'accuracy']
    
    for metric in metrics_to_compare:
        if all(metric in results for results in evaluation_results.values()):
            print(f"\n📈 {metric.upper()} Karşılaştırması:")
            
            # Veri hazırla
            models = list(evaluation_results.keys())
            scores = [evaluation_results[model][metric] for model in models]
            
            # Plot
            plt.figure(figsize=(12, 6))
            bars = plt.bar(models, scores, color='skyblue', alpha=0.7)
            
            # En iyi modeli vurgula
            best_idx = np.argmax(scores)
            bars[best_idx].set_color('gold')
            
            plt.title(f'Model Comparison - {metric.upper()}')
            plt.ylabel(metric.upper())
            plt.xticks(rotation=45)
            plt.grid(axis='y', alpha=0.3)
            plt.tight_layout()
            plt.show()
            
            # Sayısal değerleri yazdır
            for model, score in zip(models, scores):
                print(f"  {model:<20}: {score:.4f}")


In [ ]:
if comparison_df is not None:
    # En iyi modeli bul
    best_model_name = comparison_df.iloc[0]['Model']
    best_results = evaluation_results[best_model_name]
    
    print("🏆 EN İYİ MODEL DETAYLI ANALİZİ")
    print("="*50)
    print(f"Model: {best_model_name}")
    print(f"F1-Macro Score: {best_results['f1_macro']:.4f}")
    
    # Get test data (encoded)
    _, y_test_encoded = evaluator.prepare_test_data(X_test, y_test)
    
    # Detaylı confusion matrix
    print(f"\n📊 {best_model_name} - Detaylı Confusion Matrix:")
    cm, cm_norm = evaluator.plot_confusion_matrix(
        y_test_encoded, best_results['y_pred'], best_model_name, figsize=(14, 6)
    )
    
    # Sınıf bazında detaylı metrikler
    print(f"\n📋 Sınıf Bazında Detaylı Performans:")
    class_names = ['Poor', 'Standard', 'Good']
    
    # Classification report tablosu
    from sklearn.metrics import classification_report
    print("\nClassification Report:")
    print(classification_report(
        y_test_encoded, best_results['y_pred'], 
        target_names=class_names, digits=4
    ))
    
    # ROC Curve
    if best_results['y_pred_proba'] is not None:
        print(f"\n📈 {best_model_name} - ROC Curves:")
        evaluator.plot_roc_curves(
            y_test_encoded, best_results['y_pred_proba'], best_model_name
        )
    
    # Feature importance (eğer varsa)
    if hasattr(best_results['model'], 'feature_importances_'):
        print(f"\n🌳 {best_model_name} - Feature Importance:")
        evaluator.plot_feature_importance(
            best_results['model'], X_test.columns.tolist(), 
            best_model_name, top_n=20
        )

In [ ]:
if evaluation_results and comparison_df is not None:
    print("🔍 HATA ANALİZİ")
    print("="*30)
    
    # Get test data (encoded)  
    X_test_processed, y_test_encoded = evaluator.prepare_test_data(X_test, y_test)
    
    # En iyi modelin hata analizi
    best_model_name = comparison_df.iloc[0]['Model']
    best_results = evaluation_results[best_model_name]
    
    y_pred = best_results['y_pred']
    
    # Yanlış tahminler
    wrong_predictions = y_test_encoded != y_pred
    n_wrong = wrong_predictions.sum()
    
    print(f"📊 {best_model_name} Hata İstatistikleri:")
    print(f"  Toplam test örneği: {len(y_test_encoded)}")
    print(f"  Yanlış tahmin sayısı: {n_wrong}")
    print(f"  Hata oranı: {n_wrong/len(y_test_encoded)*100:.2f}%")
    
    # Hata dağılımı matrisi
    class_names = ['Poor', 'Standard', 'Good']
    
    print(f"\n🎯 Gerçek vs Tahmin Hata Dağılımı:")
    error_analysis = pd.crosstab(
        pd.Series(y_test_encoded, name='Gerçek'), 
        pd.Series(y_pred, name='Tahmin'),
        margins=True
    )
    display(error_analysis)
    
    # En çok karıştırılan sınıflar
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_test_encoded, y_pred)
    
    print(f"\n❌ En Çok Karıştırılan Sınıflar:")
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            if i != j and cm[i, j] > 0:
                print(f"  {class_names[i]} -> {class_names[j]}: {cm[i, j]} örnek")
    
    # Yanlış tahmin edilen örneklerin özellik istatistikleri
    if n_wrong > 0:
        print(f"\n📈 Yanlış Tahmin Edilen Örneklerin Özellik Analizi:")
        
        # Bazı temel özelliklerde fark var mı?
        key_features = ['Age', 'Annual_Income', 'Credit_Utilization_Ratio', 'Outstanding_Debt']
        available_features = [f for f in key_features if f in X_test.columns]
        
        if available_features:
            wrong_samples = X_test[wrong_predictions]
            correct_samples = X_test[~wrong_predictions]
            
            print(f"Anahtar özelliklerin ortalamaları:")
            print(f"{'Özellik':<25} {'Doğru':<10} {'Yanlış':<10} {'Fark':<10}")
            print("-" * 55)
            
            for feature in available_features:
                if feature in X_test.columns:
                    correct_mean = correct_samples[feature].mean()
                    wrong_mean = wrong_samples[feature].mean()
                    diff = wrong_mean - correct_mean
                    
                    print(f"{feature:<25} {correct_mean:<10.2f} {wrong_mean:<10.2f} {diff:<10.2f}")


In [ ]:
if evaluation_results:
    print("📊 CV vs TEST PERFORMANCE ANALİZİ")
    print("="*45)
    
    # CV ve Test skorlarını karşılaştır
    cv_vs_test_data = []
    
    for model_name, results in evaluation_results.items():
        if results['cv_score_mean'] is not None:
            cv_vs_test_data.append({
                'Model': model_name,
                'CV_Mean': results['cv_score_mean'],
                'CV_Std': results['cv_score_std'],
                'Test_F1': results['f1_macro'],
                'Difference': results['f1_macro'] - results['cv_score_mean']
            })
    
    if cv_vs_test_data:
        cv_comparison = pd.DataFrame(cv_vs_test_data)
        
        print("📋 CV vs Test Skorları:")
        display(cv_comparison.round(4))
        
        # Görselleştirme
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        
        # CV vs Test scatter plot
        ax1.scatter(cv_comparison['CV_Mean'], cv_comparison['Test_F1'], 
                   s=100, alpha=0.7, color='blue')
        
        # Diagonal line (perfect match)
        min_val = min(cv_comparison['CV_Mean'].min(), cv_comparison['Test_F1'].min())
        max_val = max(cv_comparison['CV_Mean'].max(), cv_comparison['Test_F1'].max())
        ax1.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.7)
        
        # Model isimlerini ekle
        for _, row in cv_comparison.iterrows():
            ax1.annotate(row['Model'], 
                        (row['CV_Mean'], row['Test_F1']),
                        xytext=(5, 5), textcoords='offset points',
                        fontsize=9)
        
        ax1.set_xlabel('CV F1-Macro Score')
        ax1.set_ylabel('Test F1-Macro Score')
        ax1.set_title('CV vs Test Performance')
        ax1.grid(True, alpha=0.3)
        
        # Fark analizi
        colors = ['green' if x >= 0 else 'red' for x in cv_comparison['Difference']]
        bars = ax2.bar(cv_comparison['Model'], cv_comparison['Difference'], 
                      color=colors, alpha=0.7)
        
        ax2.axhline(y=0, color='black', linestyle='-', alpha=0.5)
        ax2.set_xlabel('Models')
        ax2.set_ylabel('Test - CV Score')
        ax2.set_title('Performance Difference (Test - CV)')
        ax2.tick_params(axis='x', rotation=45)
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Overfitting/Underfitting analizi
        print(f"\n🔍 Overfitting/Underfitting Analizi:")
        for _, row in cv_comparison.iterrows():
            diff = row['Difference']
            if diff > 0.02:
                print(f"  {row['Model']}: Test > CV (+{diff:.3f}) - Şanslı test seti?")
            elif diff < -0.02:
                print(f"  {row['Model']}: Test < CV ({diff:.3f}) - Potansiyel overfitting")
            else:
                print(f"  {row['Model']}: Stabil performans ({diff:.3f})")


In [ ]:
print("📋 FINAL ÖZET RAPORU")
print("="*60)

if evaluation_results and comparison_df is not None:
    # En iyi model
    best_model = comparison_df.iloc[0]['Model'] 
    best_score = comparison_df.iloc[0]['Test_f1_macro']
    
    print(f"🏆 EN İYİ MODEL: {best_model}")
    print(f"🎯 F1-Macro Score: {best_score:.4f}")
    
    # Performans kategorileri
    print(f"\n📊 MODEL PERFORMANS KATEGORİLERİ:")
    
    performance_thresholds = {
        'Mükemmel (>0.85)': 0.85,
        'İyi (0.75-0.85)': 0.75, 
        'Orta (0.65-0.75)': 0.65,
        'Düşük (<0.65)': 0.0
    }
    
    for category, threshold in performance_thresholds.items():
        models_in_category = []
        for _, row in comparison_df.iterrows():
            score = row['Test_f1_macro']
            if category == 'Mükemmel (>0.85)' and score > 0.85:
                models_in_category.append(f"{row['Model']} ({score:.3f})")
            elif category == 'İyi (0.75-0.85)' and 0.75 <= score <= 0.85:
                models_in_category.append(f"{row['Model']} ({score:.3f})")
            elif category == 'Orta (0.65-0.75)' and 0.65 <= score < 0.75:
                models_in_category.append(f"{row['Model']} ({score:.3f})")
            elif category == 'Düşük (<0.65)' and score < 0.65:
                models_in_category.append(f"{row['Model']} ({score:.3f})")
        
        if models_in_category:
            print(f"  {category}:")
            for model in models_in_category:
                print(f"    • {model}")
    
    # Model seçimi önerileri
    print(f"\n💡 MODEL SEÇİMİ ÖNERİLERİ:")
    
    if best_score > 0.85:
        print(f"  ✅ {best_model} mükemmel performans gösteriyor")
        print(f"  ✅ Production'a hazır")
    elif best_score > 0.75:
        print(f"  ✅ {best_model} iyi performans gösteriyor") 
        print(f"  ⚠️  Daha fazla veri veya feature engineering ile iyileştirilebilir")
    else:
        print(f"  ⚠️  En iyi model bile {best_score:.3f} skorla düşük performans")
        print(f"  🔧 Veri kalitesi, feature engineering veya model hiperparametrelerini gözden geçirin")
    
    # Ensemble önerisi
    if 'Ensemble' in evaluation_results:
        ensemble_score = evaluation_results['Ensemble']['f1_macro']
        print(f"\n🤝 ENSEMBLE MODEL:")
        print(f"  Ensemble Score: {ensemble_score:.4f}")
        if ensemble_score >= best_score:
            print(f"  ✅ Ensemble modeli kullanılabilir")
        else:
            print(f"  ⚠️  Ensemble tek modelden düşük performans")
    
    # İş kullanımı önerileri
    print(f"\n🏢 İŞ KULLANIMI ÖNERİLERİ:")
    print(f"  • Model güvenilirliği: {'Yüksek' if best_score > 0.80 else 'Orta' if best_score > 0.70 else 'Düşük'}")
    print(f"  • Önerilen kullanım: {'Otomatik karar' if best_score > 0.85 else 'Karar destek sistemi' if best_score > 0.75 else 'Manuel inceleme gerekli'}")
    print(f"  • Risk seviyesi: {'Düşük' if best_score > 0.85 else 'Orta' if best_score > 0.75 else 'Yüksek'}")

print(f"\n✅ Model değerlendirmesi tamamlandı!")
print(f"📁 Sonuçlar '../models/evaluation_results.pkl' dosyasına kaydedildi")


In [ ]:
print("💾 SON İŞLEMLER")
print("="*30)

# Evaluation sonuçlarını kaydet
if evaluation_results and comparison_df is not None:
    # Pickle olarak kaydet
    evaluator.save_evaluation_results(evaluation_results, comparison_df)
    
    # CSV olarak da kaydet
    comparison_df.to_csv('../models/model_comparison.csv', index=False)
    print("✅ Model karşılaştırması CSV olarak kaydedildi")
    
    # Summary raporu text olarak kaydet
    summary_text = evaluator.generate_summary_report_text(evaluation_results, comparison_df)
    with open('../models/evaluation_summary.txt', 'w', encoding='utf-8') as f:
        f.write(summary_text)
    print("✅ Özet raporu text olarak kaydedildi")

# Memory temizleme
import gc
gc.collect()

print(f"\n🎉 Model Evaluation tamamlandı!")
print(f"📊 Toplam {len(evaluation_results)} model değerlendirildi")
print(f"🏆 En iyi model: {comparison_df.iloc[0]['Model'] if comparison_df is not None else 'N/A'}")
print(f"📁 Tüm sonuçlar '../models/' dizininde kaydedildi")
